In [1]:
from openai import OpenAI
import pickle
import time
import random
import ast
random.seed(0)

In [2]:
def process_properties(tables_dict, captions_dict, api_key):
    '''
    Converts 2D list table into list of tuples format
    table = [["A", "B", "C"], 
             ["1", "2", "3"]]
    table_tuples = [("A", "B", "C"), ("1", "2", "3")]
    
    We format it this way because:
    - Maintains data structure more explicitly
    - Makes it easier to process structured data
    - Preserves relationship between elements in each row
    '''
    client = OpenAI(
    api_key=api_key,
    base_url="https://api.deepseek.com/v1"  # DeepSeek's API endpoint
    )
    outputs = {}
    retry_count = 10
    wait_time = 3  # Seconds to wait between retries
    
    for key, table in tables_dict.items():
        pii, t_idx = key
        caption = captions_dict[key]
        # Convert each row to tuple and create list of tuples
        table_tuples = [tuple(str(cell) for cell in row) for row in table]
        caption = captions_dict.get(key, "")
        
        example_table_1 = [('Glasses', 'B=H/Kc (m-1/2)', 'P C  (N)', 'P C R*  (N)', 'P C  (N)', 'P C L*  (N)'), ('Float glass', '8500', '0.25-0.50', '0.30', '1-2', '-'), ('SLS1', '7500', '0.10-0.25', '0.70', '0.25-0.50', '1.18'), ('SLS2', '7350', '0.10-025', '0.68', '0.25-0.50', '1.17')]

        example_output_1 = """[]"""

        example_table_2 = [('Rare earth (RE) ion (radius, A)', '60Si20Mg20RE (100- x)OxN', '60Si20Mg20RE (100- x)OxN', '60Si20Mg20RE (100- x)OxN', '55Si25Al20RE (100- x)OxN', '55Si25Al20RE (100- x)OxN', '55Si25Al20RE (100- x)OxN'), ('Rare earth (RE) ion (radius, A)', 'Elastic modulus (GPa)', 'Hardness (GPa)', 'Nitrogen content (eq.%)', 'Elastic modulus (GPa)', 'Hardness (GPa)', 'Nitrogen content (eq.%)'), ('La3+ (1.05)', '146.5', '10.9', '23', '131.6', '10.3', '21'), ('Gd3+ (0.94)', '159.4', '12.3', '25', '144.2', '11.2', '19')]

        example_output_2 = """[[('S0022309303007713_0_2_1', "Young's modulus", 146.5, 'GPa')], [('S0022309303007713_0_2_4', "Young's modulus", 131.6, 'GPa')], [('S0022309303007713_0_3_1', "Young's modulus", 159.4, 'GPa')], [('S0022309303007713_0_3_4', "Young's modulus", 144.2, 'GPa')], [('S0022309303007713_0_2_2', 'Vickers hardness', 10.9, 'GPa')], [('S0022309303007713_0_2_5', 'Vickers hardness', 10.3, 'GPa')], [('S0022309303007713_0_3_2', 'Vickers hardness', 12.3, 'GPa')], [('S0022309303007713_0_3_5', 'Vickers hardness', 11.2, 'GPa')]]"""

        example_table_3 = [('Nr. exp', 'SiO2 (wt%)', 'K2O (wt%)', 'K2O (mol%)', 'Fe2O3 (wt%)', 'T g (K)'), ('2', '71.77 (71.27)', '28.13 (28.57)', '20.0', '0.10 (0.16)', '741'), ('3', '71.77 (72.06)', '28.13 (27.78)', '20.0', '0.10 (0.16)', 'N.A.'), ('4', '65.61 (65.25)', '34.29 (34.60)', '25.0', '0.10 (0.15)', '745'), ('5', '60.90 (60.39)', '39.00 (39.46)', '29.0', '0.10 (0.14)', 'N.A.'), ('6', '60.43 (59.76)', '39.47 (40.08)', '29.4', '0.10 (0.16)', '773')]

        example_output_3 = """[[('S0022309307006928_0_1_5_2', 'Glass transition temperature', 741.0, 'K')], [('S0022309307006928_0_3_5_4', 'Glass transition temperature', 745.0, 'K')], [('S0022309307006928_0_5_5_6', 'Glass transition temperature', 773.0, 'K')]]"""
        
        # Combine examples with their metadata
        examples = [
            {
                "pii": "S0022309302019373",
                "t_idx": 2,
                "table": example_table_1,
                "output": example_output_1,
                "caption": """Brittleness (B), experimental and calculated radial-crack-initiation loads (PCR and PCR* respectively) and experimental and calculated lateral crack threshold loads (PCL and PCL* respectively) for the SLS glasses"""
            },
            {
                "pii": "S0022309303007713",
                "t_idx": 0,
                "table": example_table_2,
                "output": example_output_2,
                "caption": "Influence of composition on elastic modulus and hardness"
            },
            {
                "pii": "S0022309307006928",
                "t_idx": 0,
                "table": example_table_3,
                "output": example_output_3,
                "caption": "Compositions of K2O-SiO2 glasses used for calibration in the evaporation experiments"
            }
        ]

        examples_text = "\n\n\n".join([
    f"Example {i+1}, Table Caption: ({ex['caption']}):\n"
    f"PII: {ex['pii']}, Table Index: {ex['t_idx']}\n"
    f"Table: {ex['table']}\n"
    f"Output: {ex['output']}"
    for i, ex in enumerate(examples)
    ])

    #Three line gap has been maintained between each examples
        
        
        prompt = f"""Extract properties following exact format of the given example, and return only the structured data.
If no property data is found, return empty list [].

Properties to extract:
property_names = ['Density', 'Glass transition temperature', 'Refractive index', 'Abbe value', "Young's modulus", 'Shear modulus', 
'Vickers hardness', 'Poisson ratio', 'Fracture toughness', 'Crystallization temp', 'Melting temp', 'Electric conductivity', 
'Softening Point (Temperature)', 'Annealing Point (Temperature)', 'Thermal expansion coefficient', 'Liquidus temperature', 'Bulk modulus', 'Activation energy']
- Use these names exactly in the output tuples.
- For "Young's modulus", include both Young's modulus and Elastic Constant.
- For 'Vickers hardness', extract all hardness types (e.g., Vickers, Knoop, etc.)

Examples, table with metadata and desired output:
{examples_text}

ID construction instructions:
- For column-oriented tables:
  - Properties are arranged in columns; values are read across.
  - ID format: id = 'pii_t_idx_i_j_Material_ID', if Material_ID is present else id = 'pii_t_idx_i_j',
where i is the row index and j is the column index containing the property values.

- For row-oriented tables:
  - Properties are arranged in rows; values are read down.
  - ID format: id = 'pii_t_idx_i_j_Material_ID', if Material_ID is present else gid = 'pii_t_idx_i_j',
where i is the row index and j is the column index containing the property values.

-For extracting the unit of the corresponding property, normalize the units according to these standards:
    
    Density: normalize to g/cm3, mg/m3, kg/m3, g/cm, or lb/in3
    Example: if you find "gm/cc" or "g cm-3", normalize to "g/cm3"
    
    Temperature-related properties (Glass transition temperature, Crystallization temp, Melting temp, Softening Point, Annealing Point, Liquidus temperature): normalize to degC or K
    Example: if you find "°C" or "C", normalize to "degC"
    
    Properties without units (Refractive index, Abbe value, Poisson ratio, Dielectric constant): use None
    Example: always use None.
    
    Elastic moduli (Young's modulus, Shear modulus, Bulk modulus): normalize to GPa, MPa, Pa, psi, dyn/cm2, or kb
    Example: if you find "G P a", normalize to "GPa"
    
    Hardness: This requires special handling based on the type:
    
    1. For Vickers Hardness (VH):
       - Preferentially normalize to GPa, MPa, Pa, or HV
       - If no unit specified, assume HV
       - Convert units like kg/mm2 or kgf/mm2 to GPa
    
    2. For other hardness scales, maintain the original scale but normalize the format:
       - Vickers variations: HV, HV1, HV2, HV3, HV5, HV10
       - Rockwell scales: HRC, HRA, HRB, HR, HRH, HRL, HCi, Hc
       - Other scales: HK (for Knoop), HB (for Brinell), Shore D, Shore A, Shore, Mohs
       Example: if you find "Hv", normalize to "HV"
    
    Fracture toughness: normalize to MPam1/2
    Example: if you find "MPa·m1/2" or "MPa(m)1/2", normalize to "MPam1/2"
    
    Electric conductivity: normalize to O-1 cm-1, O-1 m-1, or (MO m)-1
    Example: if you find "Scm-1" or "S/cm", normalize to "O-1 cm-1"
    
    Thermal expansion coefficient: normalize to degC-1 or K-1
    Example: if you find "/degC" or "C-1", normalize to "degC-1"
    
    Activation energy: normalize to kJ/mol, eV/at, kcal/mol, J/mol, or eV
    Example: if you find "kJmol-1" or "kJ mol-1", normalize to "kJ/mol"
    
    General rules for normalization:
    1. Remove whitespace in units
    2. Convert units to standard forms or closest acceptable values.
    3. If unit cannot be normalized, retain original unit.

- All indices start from 0
- For this table, pii={pii} and t_idx={t_idx}, caption={caption}

Table to process from which composition is to be extracted in the desired format:
{table_tuples}

Note: Follow the same format as the example. Each property should be a tuple with (id, propert_name, value, unit), with values as floats. Remove tuples with value = 0."""

        for attempt in range(retry_count):
            try:
                response = client.chat.completions.create(
                    model="deepseek-chat",
                    messages=[
                        {"role": "system", "content": "As a materials science expert skilled in extracting information from tables, your job is to extract the desired properties of materials from tables in the instructed format. Return only the structured data list, no additional text."},
                        {"role": "user", "content": prompt}
                    ],
                    temperature=0,
                    max_tokens=8192
                )
                
                outputs[key] = ast.literal_eval(response.choices[0].message.content.strip())
                pickle.dump(outputs, open('property_test_results_deepseek_temporary.pkl', 'wb'))
                break


                
            except Exception as e:
                if any(msg in str(e).lower() for msg in ["rate limit", "429", "connection", "timeout", "busy", "expecting value"]):
                    if attempt < retry_count - 1:
                        time.sleep(wait_time)
                        print(e)
                        print(f"Rate limit hit, retrying in {wait_time} seconds (attempt {attempt+1}/{retry_count})...") # Informative message
                    else:
                        outputs[key] = f"Error: Rate limit reached after {retry_count} attempts."
                        print(outputs[key])  # Print the final error message
                        break
                else:
                    outputs[key] = f"Error: {str(e)}"
                    print(outputs[key])  # Print the error for debugging
                    break  # Do not retry for other errors
    
    return outputs

In [1]:
random.seed(0)
# tables = pickle.load(open('tables_val_mini_dict.pkl', 'rb'))
# captions = pickle.load(open('captions_val_mini_dict.pkl', 'rb'))
tables = pickle.load(open('test_tables_dict.pkl', 'rb'))
captions = pickle.load(open('test_captions_dict.pkl', 'rb'))
start_time = time.time()
api_key = ''
results = process_properties(tables, captions, api_key)
end_time = time.time()
print(f'Time required for processing {len(tables)} tables is {end_time -  start_time} seconds')
# pickle.dump(results, open('composition_results.pkl', 'wb'))

In [4]:
pickle.dump(results, open('property_test_results_deepseek__chat_v3.pkl', 'wb'))
print(len(results))

368
